In [ ]:
# @title Setup — Run this first
import os

if not os.path.exists('/content/MS_spectra_encoding'):
    !git clone https://github.com/Alexander-Sol/MS_spectra_encoding.git /content/MS_spectra_encoding

%cd /content/MS_spectra_encoding

# Install packages not pre-installed on Colab
!pip install -q pyteomics==4.6.1 spectrum-utils==0.4.2 rapidhash==0.1.0 mmh3

# The DDA data file is force-committed, so it clones with the repo.
# gdown fallback (uncomment + fill FILE_ID if the committed copy is unavailable):
# !gdown <FILE_ID> -O Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML


## Goals:

In this notebook, we'll be exploring the following three core concepts:

1. **What a peptide feature is** — the LC-MS map (MS1 signal over retention time), the isotope envelope (isotopes spaced `1/z` in m/z), bell-shaped elution, charge states, and overlapping features; why feature detection is framed as 3D segmentation.
2. **DeepIso = image encoding** — binning the LC-MS map into a fixed-resolution RT × m/z grayscale image (m/z → 0.01, RT → 1 scan, intensity → 0–255), overlaid with the [15 × 211] scan framing window.
3. **PointIso = point-cloud encoding** — representing the same feature as arbitrary-precision (RT, m/z, intensity) triplets; a side-by-side sparsity/memory comparison against the image, closing with the 4D ion-mobility (TimsTOF) extension.

In [ ]:
# @title Run this cell to import all necessary packages
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pyteomics import mzml

# util.py is under active development, so we force-reload it here to pick up any edits
import util
importlib.reload(util)
from util import *

## 5.1 Concept 1: What is a peptide feature?

Notebooks 01–04 lived inside a **single spectrum** — one MS2 scan, its peaks, and the peptide that produced them. Feature detection asks a different question, one scan up: *where in the whole LC-MS run does a given peptide live?* To answer it we stop looking at individual spectra and start looking at the **LC-MS map**.

### The LC-MS map

During an LC-MS/MS run, the mass spectrometer records an **MS1 survey scan** every fraction of a second while peptides elute off the liquid-chromatography (LC) column. Stack those MS1 scans in acquisition order and you get a 2D map with three axes of information:

- **Retention time (RT)** — *when* a peak was seen (which MS1 scan). Rows of the map.
- **m/z** — *where* on the mass axis the peak sits. Columns of the map.
- **Intensity** — *how much* signal, at each (RT, m/z) location. The value at each cell.

This (RT × m/z → intensity) surface is the raw material for feature detection. It is genuinely new territory for this notebook series: notebooks 01–03 worked with MS2 fragment spectra, and notebook 04 stitched neighboring scans into an augmented spectrum, but none of them looked at the *elution* dimension the way a feature detector must.

### A peptide feature

A **peptide feature** is the complete MS1 footprint of one peptide (at one charge state) on that map. It has a characteristic shape in each direction:

- **Along m/z — the isotope envelope.** A peptide is not a single peak. Because carbon, nitrogen, etc. have naturally occurring heavy isotopes (mostly ¹³C), the same peptide appears as a *series* of peaks: the monoisotopic peak (all light isotopes) plus +1, +2, +3… isotopologues of decreasing intensity. Crucially, the instrument measures **m/z**, not mass, so successive isotopes — one extra neutron ≈ 1 Da apart in mass — are spaced **`1/z` in m/z**, where `z` is the charge. A +2 ion shows isotopes 0.5 m/z apart; a +3 ion, 0.333 m/z apart. **That `1/z` spacing is the fingerprint that tells you a cluster of peaks is one peptide, and reveals its charge.**
- **Along RT — bell-shaped elution.** A peptide does not elute instantly. It washes off the LC column over a few seconds to tens of seconds, so its intensity rises, peaks (the *apex*), and falls — an approximately Gaussian **elution profile** across consecutive MS1 scans.
- **Charge state.** Peptides pick up protons and appear at charges of roughly **+1 to +9**. Charge sets the isotope spacing (`1/z`) and must be inferred as part of detection.

Put together, a feature is a small 2D "blob": a few isotope peaks wide (spaced `1/z`), several scans tall (the elution bell), brightest at its apex.

### Our real example

Throughout this notebook we use one clean, real feature from the committed DDA file, chosen in advance for how textbook it looks:

| Property | Value |
|---|---|
| Precursor m/z | **1158.22** |
| Charge `z` | **+3** → isotope spacing `1/z ≈ 0.333` m/z |
| Retention time | **~13.06 min** (elution apex at MS1 scan 5450) |
| PSM scan | 5453 |

Four well-resolved isotopes sit at 1158.22, 1158.55, 1158.88, 1159.22 — each step ≈ 0.334 m/z, matching `1/z` to within 0.002. We load a small region of the map around this feature (±0.5 min RT, ±3 m/z) in the next cell.

### Why detection is hard — and why it is framed as segmentation

Real runs are crowded. Many peptides elute at once, so features **overlap**: envelopes interleave on the m/z axis and elution bells pile up on the RT axis. Detecting a feature therefore means more than "find a bright spot" — it means deciding, for *every* point on the map, **which feature (if any) that point belongs to**. That is exactly a **segmentation** problem, over a 3D (RT, m/z, intensity) space.

Both models we study frame detection this way. Where they differ — and the whole point of this notebook — is in **how they encode the LC-MS map** so a neural network can segment it:

- **DeepIso (2019)** renders the map as a fixed-resolution **grayscale image** and classifies pixels.
- **PointIso (2021)** keeps the map as a sparse **point cloud** of real peaks and segments points directly.

We treat both networks (CNN/RNN for DeepIso, PointNet + dual-attention for PointIso) as a **black box** and focus entirely on the encoding — the same approach notebooks 01–04 took. Concept 2 builds the image; Concept 3 builds the point cloud and compares them.

In [ ]:
# --- The chosen real feature (these constants are reused throughout the notebook) ---
MZML_PATH = "Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML"

FEATURE_MS2_SCAN = 5453        # PSM/MS2 scan number (from Data/ScanNumsWPsms.txt)
FEATURE_MZ       = 1158.2193   # precursor m/z
FEATURE_CHARGE   = 3           # charge z  ->  isotope spacing 1/z ~= 0.333 m/z
FEATURE_RT       = 13.065      # precursor retention time (min)

# LC-MS region around the feature: +/-0.5 min in RT, +/-3 m/z (fully contains the envelope).
RT_WINDOW = (FEATURE_RT - 0.5, FEATURE_RT + 0.5)   # (12.565, 13.565) min
MZ_WINDOW = (FEATURE_MZ - 3.0, FEATURE_MZ + 3.0)   # (1155.219, 1161.219) m/z

# Load every real MS1 peak in that window: raw (scan, RT, m/z, intensity) points.
points = get_lcms_map_region(MZML_PATH, RT_WINDOW, MZ_WINDOW)

print(f"Loaded {len(points)} raw peaks across {points['scan'].nunique()} MS1 scans")
print(f"  RT range : {points['rt'].min():.3f} - {points['rt'].max():.3f} min")
print(f"  m/z range: {points['mz'].min():.3f} - {points['mz'].max():.3f}")
points.head()

In [ ]:
# The raw feature: every real MS1 peak at its measured (m/z, RT, intensity).
# No binning, no encoding decisions yet -- this is the LC-MS map region as recorded.
# Rotate the interactive plot to see the feature's structure:
#   * along m/z  -> the isotope envelope (peaks spaced 1/z ~= 0.333 for this z=3 ion)
#   * along RT   -> the bell-shaped elution profile, brightest at the apex
raw_cloud = points[['rt', 'mz', 'intensity']].to_numpy()
fig = plot_lcms_point_cloud(
    raw_cloud,
    title=f'Raw feature - precursor {FEATURE_MZ} m/z (z={FEATURE_CHARGE}) near RT {FEATURE_RT} min',
)
fig.show()

## 5.2 Concept 2: DeepIso — encoding the map as an image

The raw feature we just plotted is a scatter of real peaks at their measured, arbitrary-precision coordinates. A convolutional neural network cannot ingest that directly — a CNN expects a **regular grid of pixels**, like a photograph. DeepIso (Zohora et al., 2019) makes the LC-MS map look exactly like a grayscale image, and then treats feature detection as an image-processing problem.

### Binning the map onto a fixed grid

The trick is to lay a fixed grid over the (RT × m/z) map and drop every peak into a cell:

- **Rows = retention time, at `RT → 1 MS-scan`.** Each MS1 scan becomes exactly one row. Row 0 is the earliest scan in the window, and moving down a row means stepping to the next scan in acquisition order. Time is quantized to the instrument's scan rate.
- **Columns = m/z, at `m/z → 0.01`.** The m/z axis is chopped into fixed **0.01-wide bins**. Every peak is snapped to the bin it lands in; two peaks in the same (scan, m/z-bin) cell have their intensities summed.
- **Cell value = intensity, squashed to `0–255`.** Each cell's summed intensity is rescaled to an 8-bit grayscale "pixel", where 0 is empty and 255 is the brightest signal in the window — literally a black-and-white image.

After this step the feature is no longer a list of points; it is a small picture. The isotope envelope becomes a row of bright vertical streaks spaced `1/z ≈ 0.333` m/z (≈ 33 columns) apart, and the elution bell becomes those streaks brightening then fading as you scan down the rows.

### The [15 × 211] framing window

DeepIso does not classify the whole map at once. It slides a fixed-size **framing window of [15 scans × 211 m/z bins]** across the image and asks the network to label the pixels inside each frame. That size is deliberate:

- **15 scans tall** comfortably spans one elution bell (a feature is only a handful of scans wide at the apex).
- **211 bins wide** is `211 × 0.01 ≈ 2.11 m/z` — wide enough to hold a full isotope envelope for the charge states DeepIso targets.

We overlay this frame on the heatmap below so you can see how a single "look" of the model covers our feature.

### Fixed precision is the whole point

Notice what the grid costs us. The moment we choose `m/z → 0.01` and `RT → 1 scan`, the resolution is **fixed**: two isotopes closer than 0.01 m/z would collapse into the same column, and the exact measured m/z (good to four decimals) is thrown away — rounded to the nearest bin. The grid also exists *everywhere*, whether or not a real peak is present, so the overwhelming majority of the 15 × 211 ≈ 3,000 pixels in a frame are zero. That wasted, mostly-empty grid is exactly the weakness PointIso fixes in Concept 3.

As with notebooks 01–04, we treat the DeepIso network itself (a CNN that scans the frame plus an RNN that stitches detections across scans) as a **black box**. Our job is only to build the image it would eat. The next cell does that on our real feature.

In [ ]:
# DeepIso encoding: bin the LC-MS map region onto a fixed grid ->
#   rows  = MS1 scans (RT -> 1 scan)
#   cols  = 0.01-wide m/z bins (m/z -> 0.01)
#   value = summed intensity, squashed to an 8-bit grayscale pixel (0-255)
# The result is literally a small grayscale image of the feature.
image = build_lcms_image(points, mz_bin=0.01, mz_range=MZ_WINDOW)

grid = image['image']
n_scans, n_mz_bins = grid.shape
filled = int((grid > 0).sum())
print(f'DeepIso image: {n_scans} scans x {n_mz_bins} m/z bins = {grid.size} pixels')
print(f'  filled pixels: {filled} ({100 * filled / grid.size:.1f}%)  ->  the rest are empty grid')
print(f'  pixel range  : {int(grid.min())} - {int(grid.max())} (intensity {image["max_intensity"]:.3g} -> 255)')

# Show the image as an RT x m/z heatmap, overlaying the DeepIso [15 scans x 211 m/z bins]
# framing window (centered on the brightest pixel) so you can see one "look" of the model.
fig = plot_lcms_heatmap(
    image,
    title='DeepIso image of the feature (0.01 m/z bins) - [15 x 211] frame overlaid',
    frame=True,
)
fig.show()

## 5.3 Concept 3: PointIso — encoding the map as a point cloud

The DeepIso image works, but look again at what it cost. To feed a CNN we laid a dense grid over the map, quantized m/z to 0.01 and RT to one scan, rescaled every intensity to an 8-bit pixel, and materialized a cell for *every* (scan, m/z-bin) location — even though almost all of them are empty. PointIso (Zohora et al., 2021) throws that grid away and asks a simpler question: **why store empty space at all?**

### The (RT, m/z, intensity) triplet point cloud

An LC-MS map is not really a picture — it is a **sparse list of real peaks**. So PointIso encodes the map as exactly that: a **point cloud**, where each detected MS1 peak is a single point with three coordinates,

$$\text{point} = (\text{RT},\; m/z,\; \text{intensity}).$$

That is all. There is no grid, no background, no filler cell. The map is just the set of points that were actually measured. This is the same representation used for 3D shapes in computer graphics (a cloud of surface points), which is why PointIso can borrow a network — **PointNet** — designed to consume point clouds directly.

### Arbitrary precision — nothing is rounded away

Because a point stores its own coordinates, there is no bin to snap to. The measured m/z is kept to its full precision (up to four decimals), and RT and intensity are kept as recorded. The `1/z ≈ 0.333` m/z isotope spacing that DeepIso approximated as "≈ 33 columns" is now preserved exactly, at whatever resolution the instrument delivered. Nothing is quantized, summed into a shared cell, or squashed to 0–255. **Precision is arbitrary rather than fixed** — the encoding no longer imposes a resolution ceiling on the data.

### A non-overlapping window

DeepIso slid a [15 × 211] frame across the image with overlap, re-examining pixels in several frames. PointIso instead partitions the map into a fixed **2.0 m/z × 15 scans** window that tiles the map **without overlap** — each region of the map is looked at once. The window is sized to hold a full isotope envelope (2.0 m/z spans the isotopes of the charge states it targets) and one elution bell (15 scans), just like DeepIso's frame, but because the points inside carry their own coordinates the window needs no internal grid.

### Still a segmentation problem — the network stays a black box

The task is unchanged from Concept 1: decide, for every point, **which feature it belongs to**. On a point cloud this is **point-wise segmentation** — PointNet plus a dual-attention module assigns each point to a feature (or to background), and a second stage groups the segmented points into charge-resolved features. As throughout notebooks 01–04, we treat that network as a **black box**; our job is only to build the point cloud it consumes.

The next cell builds that point cloud for the *same* real feature we imaged in Concept 2. After that, we put the two encodings side by side — counting filled vs. empty pixels and bytes — to make the sparsity/memory trade-off concrete, and close with the 4D ion-mobility extension that the point cloud scales to but the image does not.

In [ ]:
# PointIso encoding: keep the LC-MS map region as a sparse point cloud ->
#   each real MS1 peak = one (RT, m/z, intensity) triplet at its measured coordinates.
# No grid, no binning, no 0-255 rescaling -- full precision, only real peaks stored.
cloud = lcms_point_cloud(points, mz_range=MZ_WINDOW)

print(f'PointIso point cloud: {len(cloud)} real peaks, each a (RT, m/z, intensity) triplet')
print(f'  m/z kept to full precision (e.g. {cloud[0, 1]:.4f}) -- no 0.01 bin snapping')
print(f'  RT range : {cloud[:, 0].min():.3f} - {cloud[:, 0].max():.3f} min')
print(f'  m/z range: {cloud[:, 1].min():.4f} - {cloud[:, 1].max():.4f}')

# Show the same feature as an interactive 3D scatter of its real peaks. Rotate to see the
# isotope envelope along m/z (spacing 1/z ~= 0.333) and the elution bell along RT -- the
# same structure as the DeepIso image, but stored as points instead of a dense grid.
fig = plot_lcms_point_cloud(
    cloud,
    title=f'PointIso point cloud of the feature - precursor {FEATURE_MZ} m/z (z={FEATURE_CHARGE})',
)
fig.show()

### Image vs. point cloud — the sparsity/memory trade-off

We now have the *same* feature encoded two ways: the DeepIso **image** (`image`, from Concept 2) and the PointIso **point cloud** (`cloud`, from Concept 3). Putting them side by side is the core takeaway of the whole notebook.

The tension is this. An LC-MS window is almost entirely **empty** — a peptide is a handful of isotope peaks over a few scans, and everything around it is background. The point cloud stores *only* the real peaks, so its size is set by the data. The image instead materializes a cell for **every** (scan, m/z-bin) location in the window, real peak or not, so its size is set by the chosen **resolution** — not by how much signal is actually there.

Two consequences fall out, and the next cell measures both on our real feature:

- **Sparsity.** Even at DeepIso's coarse `0.01` m/z bins, the vast majority of the image's pixels are zero — the grid is spent almost entirely on empty space, while the point cloud has no empty entries at all.
- **Resolution cost.** The image's memory is tied to its bin width. Ask for finer m/z resolution and the pixel count grows **10× per decimal place** — `0.01 → 0.001 → 0.0001` takes a ~2 m/z envelope from thousands of pixels toward the paper's ~300,000. The point cloud never changes: the same real peaks are stored regardless of resolution, at full precision. This is exactly why the image *loses* precision (it must pick a bin) while the point cloud *keeps* it (it stores coordinates), and why the point cloud is the encoding that scales to the 4D extension below.

In [ ]:
# Side-by-side: the SAME window (`image` from Concept 2, `cloud` from Concept 3),
# counted two ways -- filled vs. empty pixels, and bytes on disk.
fp = lcms_encoding_footprint(image, cloud)

print("Same LC-MS window, two encodings")
print("=" * 46)
print(f"{'':22}{'DeepIso image':>14}{'':>2}{'PointIso cloud':>0}")
print(f"{'stored cells / points':22}{fp['n_pixels']:>14,}{'':>2}{fp['n_points']:>14,}")
print(f"{'  of which real peaks':22}{fp['n_filled']:>14,}{'':>2}{fp['n_points']:>14,}")
print(f"{'  empty grid':22}{fp['n_empty']:>14,}{'':>2}{0:>14,}")
print(f"{'sparsity (fraction 0)':22}{fp['sparsity']:>13.1%}{'':>3}{'--':>14}")
print(f"{'bytes':22}{fp['image_bytes']:>14,}{'':>2}{fp['point_cloud_bytes']:>14,}")
print("=" * 46)
print(f"At 0.01 m/z bins the image is {fp['sparsity']:.1%} empty and costs "
      f"{fp['memory_ratio']:.1f}x the point cloud's memory.")
print()

# Now the punchline: shrink the m/z bin toward the paper's fine resolution. The image
# balloons 10x per decimal (a cell exists everywhere), while the point cloud NEVER
# changes -- it only ever stores the real peaks.
print("As we ask the image for finer m/z resolution:")
print(f"{'m/z bin':>10}{'image pixels':>16}{'image bytes':>14}{'sparsity':>11}{'vs cloud':>11}")
for mz_bin in (0.01, 0.001, 0.0001):
    img_b = build_lcms_image(points, mz_bin=mz_bin, mz_range=MZ_WINDOW)
    fpb = lcms_encoding_footprint(img_b, cloud)
    print(f"{mz_bin:>10}{fpb['n_pixels']:>16,}{fpb['image_bytes']:>14,}"
          f"{fpb['sparsity']:>10.2%}{fpb['memory_ratio']:>10.0f}x")
print(f"{'point cloud':>10}{fp['n_points']:>16,}{fp['point_cloud_bytes']:>14,}"
      f"{'--':>11}{'1x':>11}  (unchanged at every resolution)")

### The 4D extension: adding ion mobility

The point cloud earns its keep the moment the instrument adds a dimension. On a **TimsTOF** (trapped ion mobility spectrometry), ions are separated not only by m/z and retention time but also by their **ion mobility** — how quickly they drift through a gas, which depends on their collision cross-section (shape and size). This is reported as the **inverse reduced ion mobility, `1/K₀`**. A peptide feature now spreads out over m/z, RT, *and* `1/K₀`, giving a cleaner separation of co-eluting, overlapping ions than m/z and RT alone.

Encoding this is where the two representations part ways for good:

- **Point cloud (PointIso).** Each real peak simply grows from a 3-tuple `(RT, m/z, intensity)` to a **4-tuple** `(RT, m/z, intensity, 1/K₀)`. You append one number per point. The number of stored points is still set by how many real peaks there are — the encoding's size tracks the *data*, exactly as before. This is precisely the 4D extension PointIso targets.

- **Image (DeepIso).** Adding a coordinate means the 2D grid must become a **3D volume**: every existing `(scan, m/z-bin)` pixel is replicated across a new stack of `1/K₀` bins. The pixel count is *multiplied* by the number of mobility bins, so the memory cost grows **multiplicatively** with each dimension — and since the map was already ~99% empty in 2D, the emptiness only compounds. A grid pays for resolution in every dimension at once, whether or not there is signal there.

That is the whole argument in one line: **the image's cost scales with resolution and dimensionality; the point cloud's cost scales with the data.** The sparse encoding is the one that generalizes — from 3D LC-MS maps to 4D ion-mobility data, and to whatever dimension an instrument adds next.

## What we learned

This notebook moved one level up from the single MS2 spectra of notebooks 01–04 to the **LC-MS map** — MS1 signal over retention time — and asked how two feature-detection models *encode* that map for a neural network.

1. **A peptide feature is a shape on the map.** One peptide (at one charge) leaves an isotope envelope spaced `1/z` in m/z and a bell-shaped elution profile over RT. Because real runs are crowded with overlapping features, detection is framed as **3D segmentation**: decide, for every point, which feature it belongs to.

2. **DeepIso encodes the map as a fixed-resolution image.** Bin RT → 1 scan and m/z → 0.01, squash intensity to a 0–255 pixel, and slide a [15 × 211] frame across the grid. This makes the map look like a photograph a CNN can read — but the resolution is *fixed* (exact m/z is rounded away) and a cell exists everywhere, so the image is ~99% empty.

3. **PointIso encodes the map as a point cloud.** Keep every real peak as an arbitrary-precision `(RT, m/z, intensity)` triplet — no grid, no binning, no rounding. The same feature is stored in a fraction of the memory, at full precision, and PointNet segments the points directly.

**The one-line takeaway:** the image's cost scales with **resolution and dimensionality**; the point cloud's cost scales with the **data**. That is why the sparse encoding keeps full precision, wastes no memory on empty space, and extends cleanly to the 4D TimsTOF ion-mobility case where the image would balloon into a mostly-empty volume. Both networks stayed a black box throughout — as in every notebook here, the lesson is in the *encoding*, not the training.

## Appendix: Function Definitions

Below are the definitions of the key utility functions used throughout this notebook. They live in `util.py` and are imported at the top (via `from util import *`), so you don't need to run these cells for the notebook to work — they're here purely for reference. Feel free to skip this section if you're not interested in the implementation details.

The functions come in two pairs plus a comparison helper:

- **`get_lcms_map_region`** — reads the raw `(scan, RT, m/z, intensity)` points of an LC-MS map region from the mzML file (the shared starting point for both encodings). Relies on `extract_scan_number` and `pyteomics`, also in `util.py`.
- **`build_lcms_image`** / **`plot_lcms_heatmap`** — the DeepIso *image* encoding: bin the map onto a fixed RT × m/z grid of 0–255 pixels, and draw it as a heatmap with the optional [15 × 211] framing window.
- **`lcms_point_cloud`** / **`plot_lcms_point_cloud`** — the PointIso *point-cloud* encoding: keep every real peak as an arbitrary-precision `(RT, m/z, intensity)` triplet, and draw it as a 3D scatter.
- **`lcms_encoding_footprint`** — the sparsity/memory comparison that quantifies the core takeaway (filled vs. empty pixels; image bytes vs. point-cloud bytes).

In [ ]:
def get_lcms_map_region(mzml_path, rt_range, mz_range):
    """Extract the LC-MS map region for a peptide feature as a flat point list.

    Walks the MS1 scans of an mzML file and collects every peak whose retention
    time falls in ``rt_range`` and whose m/z falls in ``mz_range``. This is the
    raw (retention time, m/z, intensity) representation of an LC-MS map region --
    the shared starting point for both the DeepIso image encoding and the
    PointIso point-cloud encoding in notebook 05.

    Parameters
    ----------
    mzml_path : str
        Path to the mzML file (e.g. the committed DDA calibration file).
    rt_range : (float, float)
        Inclusive (low, high) retention-time window in minutes.
    mz_range : (float, float)
        Inclusive (low, high) m/z window.

    Returns
    -------
    pandas.DataFrame
        One row per peak with columns ``scan`` (int MS1 scan number),
        ``rt`` (float, minutes), ``mz`` (float), and ``intensity`` (float),
        sorted by ``rt`` then ``mz``. Empty (zero-row) frame if nothing matches.
    """
    lo_rt, hi_rt = rt_range
    lo_mz, hi_mz = mz_range

    scans, rts, mzs, intensities = [], [], [], []
    with pyteomics.mzml.read(mzml_path) as spectra:
        for spectrum in spectra:
            if spectrum.get('ms level', 0) != 1:
                continue
            rt = float(spectrum['scanList']['scan'][0]['scan start time'])
            if rt < lo_rt or rt > hi_rt:
                continue
            scan = extract_scan_number(spectrum['id'])
            mz = spectrum['m/z array']
            intensity = spectrum['intensity array']
            mask = (mz >= lo_mz) & (mz <= hi_mz)
            n = int(mask.sum())
            if n == 0:
                continue
            scans.append(np.full(n, scan, dtype=int))
            rts.append(np.full(n, rt))
            mzs.append(mz[mask])
            intensities.append(intensity[mask])

    if not scans:
        return pd.DataFrame(columns=['scan', 'rt', 'mz', 'intensity'])

    df = pd.DataFrame({
        'scan': np.concatenate(scans),
        'rt': np.concatenate(rts),
        'mz': np.concatenate(mzs),
        'intensity': np.concatenate(intensities),
    })
    return df.sort_values(['rt', 'mz']).reset_index(drop=True)

In [ ]:
def build_lcms_image(points, mz_bin=0.01, mz_range=None, max_intensity=None):
    """Encode an LC-MS map region as a fixed-resolution 2D grayscale image (DeepIso).

    This is the DeepIso (Zohora et al., 2019) encoding: the LC-MS map is binned
    onto a fixed grid and every cell's intensity is squashed to a 0-255 grayscale
    "pixel", exactly as a black-and-white image. The two axes use fixed precision:

    * **RT axis (rows):** one row per MS1 scan (``RT -> 1 MS-scan``). Rows are
      ordered by increasing retention time, so row 0 is the earliest scan.
    * **m/z axis (columns):** fixed ``mz_bin`` bins (0.01 m/z in DeepIso). Peaks
      landing in the same (scan, m/z-bin) cell have their intensities summed.

    Fixing the resolution is the whole point of the comparison with PointIso: it
    forces a grid whether or not real peaks are present, so most pixels end up
    empty (see :func:`lcms_encoding_footprint`).

    Parameters
    ----------
    points : pandas.DataFrame
        Output of :func:`get_lcms_map_region` -- rows of ``scan``, ``rt``, ``mz``,
        ``intensity``.
    mz_bin : float, optional
        Width of each m/z bin (column) in m/z units. Default ``0.01`` (DeepIso).
    mz_range : (float, float), optional
        Inclusive (low, high) m/z span for the columns. Defaults to the min/max
        m/z of ``points``. Peaks outside the range are dropped.
    max_intensity : float, optional
        Intensity mapped to pixel value 255. Defaults to the max intensity in
        ``points``. Pass an explicit value to scale several windows consistently.

    Returns
    -------
    dict
        ``image`` : 2D ``uint8`` array, shape ``(n_scans, n_mz_bins)``, 0-255.
        ``rt_values`` : 1D array, the RT (min) of each row.
        ``scan_values`` : 1D int array, the MS1 scan number of each row.
        ``mz_edges`` : 1D array of length ``n_mz_bins + 1``, the m/z bin edges.
        ``mz_bin`` : the bin width used.
        ``max_intensity`` : the intensity mapped to 255.
    """
    cols = ['scan', 'rt', 'mz', 'intensity']
    if points is None or len(points) == 0:
        return {
            'image': np.zeros((0, 0), dtype=np.uint8),
            'rt_values': np.zeros(0),
            'scan_values': np.zeros(0, dtype=int),
            'mz_edges': np.zeros(0),
            'mz_bin': mz_bin,
            'max_intensity': 0.0,
        }
    df = points[cols]

    if mz_range is None:
        lo_mz, hi_mz = float(df['mz'].min()), float(df['mz'].max())
    else:
        lo_mz, hi_mz = float(mz_range[0]), float(mz_range[1])
        df = df[(df['mz'] >= lo_mz) & (df['mz'] <= hi_mz)]

    # m/z columns: fixed-width bins spanning [lo_mz, hi_mz].
    n_mz = max(1, int(np.ceil((hi_mz - lo_mz) / mz_bin)))
    mz_edges = lo_mz + mz_bin * np.arange(n_mz + 1)

    # RT rows: one per MS1 scan, ordered by retention time.
    scan_order = df[['scan', 'rt']].drop_duplicates().sort_values('rt')
    scan_values = scan_order['scan'].to_numpy(dtype=int)
    rt_values = scan_order['rt'].to_numpy(dtype=float)
    row_of_scan = {s: i for i, s in enumerate(scan_values)}
    n_scans = len(scan_values)

    grid = np.zeros((n_scans, n_mz), dtype=float)
    if len(df):
        rows = df['scan'].map(row_of_scan).to_numpy(dtype=int)
        col = np.floor((df['mz'].to_numpy() - lo_mz) / mz_bin).astype(int)
        col = np.clip(col, 0, n_mz - 1)
        np.add.at(grid, (rows, col), df['intensity'].to_numpy())

    if max_intensity is None:
        max_intensity = float(grid.max()) if grid.size else 0.0
    if max_intensity > 0:
        image = np.clip(np.round(grid / max_intensity * 255.0), 0, 255).astype(np.uint8)
    else:
        image = np.zeros_like(grid, dtype=np.uint8)

    return {
        'image': image,
        'rt_values': rt_values,
        'scan_values': scan_values,
        'mz_edges': mz_edges,
        'mz_bin': mz_bin,
        'max_intensity': max_intensity,
    }

In [ ]:
def lcms_point_cloud(points, mz_range=None):
    """Encode an LC-MS map region as a (RT, m/z, intensity) point cloud (PointIso).

    This is the PointIso (Zohora et al., 2021) encoding: instead of binning the
    LC-MS map onto a fixed grid (see :func:`build_lcms_image`), every real peak is
    kept as a single ``(retention time, m/z, intensity)`` triplet at its measured
    coordinates. There is **no binning** -- the m/z and RT axes keep their full
    precision, and only positions where a peak was actually observed are stored.

    That is the whole contrast with the DeepIso image: the image spends memory on
    a dense grid of mostly-empty pixels at a fixed resolution, while the point
    cloud stores only the handful of real peaks at arbitrary precision (and, in the
    4D TimsTOF extension, extends naturally to a fourth ion-mobility coordinate).
    See :func:`lcms_encoding_footprint` for the side-by-side sparsity/memory story.

    Parameters
    ----------
    points : pandas.DataFrame
        Output of :func:`get_lcms_map_region` -- rows of ``scan``, ``rt``, ``mz``,
        ``intensity``.
    mz_range : (float, float), optional
        Inclusive (low, high) m/z span. Defaults to the full range of ``points``.
        Peaks outside the range are dropped, matching :func:`build_lcms_image` so
        the two encodings describe exactly the same window.

    Returns
    -------
    numpy.ndarray
        Float array of shape ``(n_points, 3)`` with columns ``[rt, mz,
        intensity]``, one row per real peak, sorted by ``rt`` then ``mz``. Empty
        ``(0, 3)`` array if ``points`` is empty.
    """
    if points is None or len(points) == 0:
        return np.zeros((0, 3), dtype=float)

    df = points[['rt', 'mz', 'intensity']]
    if mz_range is not None:
        lo_mz, hi_mz = float(mz_range[0]), float(mz_range[1])
        df = df[(df['mz'] >= lo_mz) & (df['mz'] <= hi_mz)]

    if len(df) == 0:
        return np.zeros((0, 3), dtype=float)

    df = df.sort_values(['rt', 'mz'])
    return df.to_numpy(dtype=float)

In [ ]:
def lcms_encoding_footprint(image, cloud):
    """Compare the DeepIso image and PointIso point cloud on sparsity and memory.

    This is the core takeaway of notebook 05, quantified: for the *same* LC-MS
    window, how much of the fixed-resolution image is actually empty, and how many
    bytes does each encoding cost? The image (see :func:`build_lcms_image`) spends
    one pixel per (scan, m/z-bin) cell whether or not a peak is there; the point
    cloud (see :func:`lcms_point_cloud`) stores only real peaks. The punchline is
    that a high-resolution window is ~300,000 pixels but only a few thousand real
    points, so the vast majority of the image is empty space.

    Parameters
    ----------
    image : dict
        Output of :func:`build_lcms_image` (uses the ``image`` 2D ``uint8`` array).
    cloud : numpy.ndarray
        Output of :func:`lcms_point_cloud` -- ``(n_points, 3)`` float array.

    Returns
    -------
    dict
        ``n_pixels`` : total grid cells in the image (n_scans x n_mz_bins).
        ``n_filled`` : image cells with a non-zero pixel (a real peak landed there).
        ``n_empty`` : image cells that are zero (wasted on empty space).
        ``fill_fraction`` : ``n_filled / n_pixels`` (0-1); ``sparsity`` is 1 minus it.
        ``sparsity`` : fraction of the image that is empty.
        ``n_points`` : number of real peaks in the point cloud.
        ``image_bytes`` : bytes to store the dense ``uint8`` image grid.
        ``point_cloud_bytes`` : bytes to store the ``(n_points, 3)`` float triplets.
        ``memory_ratio`` : ``image_bytes / point_cloud_bytes`` (image / point cloud).
    """
    grid = np.asarray(image['image']) if isinstance(image, dict) else np.asarray(image)
    cloud = np.asarray(cloud, dtype=float)
    if cloud.ndim != 2 or cloud.shape[1] != 3:
        cloud = cloud.reshape(-1, 3) if cloud.size else np.zeros((0, 3))

    n_pixels = int(grid.size)
    n_filled = int(np.count_nonzero(grid))
    n_empty = n_pixels - n_filled
    fill_fraction = (n_filled / n_pixels) if n_pixels else 0.0

    n_points = int(len(cloud))
    image_bytes = int(grid.nbytes)
    point_cloud_bytes = int(cloud.nbytes)
    memory_ratio = (image_bytes / point_cloud_bytes) if point_cloud_bytes else float('inf')

    return {
        'n_pixels': n_pixels,
        'n_filled': n_filled,
        'n_empty': n_empty,
        'fill_fraction': fill_fraction,
        'sparsity': 1.0 - fill_fraction,
        'n_points': n_points,
        'image_bytes': image_bytes,
        'point_cloud_bytes': point_cloud_bytes,
        'memory_ratio': memory_ratio,
    }

In [ ]:
def plot_lcms_heatmap(image, title=None, frame=None):
    """Plot a DeepIso LC-MS image as an RT x m/z heatmap (intensity -> pixel 0-255).

    Renders the fixed-resolution grid produced by :func:`build_lcms_image`: each
    row is one MS1 scan (RT axis), each column is a fixed m/z bin, and the cell
    value is the 0-255 grayscale "pixel". Retention time (minutes) is used for the
    y-axis and m/z-bin centers for the x-axis so the picture is readable, while the
    pixel values are exactly what the DeepIso CNN would ingest.

    Optionally overlays the DeepIso **[15 scans x 211 m/z bins]** framing window as a
    rectangle, so the notebook can show how the model slides a fixed frame across
    the map. Prefers Plotly (interactive), falling back to matplotlib -- matching
    the rest of ``util.py``.

    Parameters
    ----------
    image : dict
        Output of :func:`build_lcms_image` (keys ``image``, ``rt_values``,
        ``mz_edges``, ``mz_bin``, ...).
    title : str, optional
        Plot title. Defaults to a description of the window size.
    frame : bool or dict, optional
        Overlay the DeepIso framing window. ``True`` draws a default
        [15 x 211] frame centered on the brightest pixel. A dict may override:

        * ``n_scans`` (int, default 15) -- frame height in MS1 scans (rows).
        * ``n_mz_bins`` (int, default 211) -- frame width in m/z bins (columns).
        * ``row0`` (int) -- top row of the frame; defaults to centering on the
          brightest pixel's row.
        * ``col0`` (int) -- left column of the frame; defaults to centering on the
          brightest pixel's column.

    Returns
    -------
    plotly.graph_objs.Figure or matplotlib.figure.Figure
        Interactive Plotly heatmap if Plotly is available, else a matplotlib figure.
    """
    grid = np.asarray(image['image'])
    rt_values = np.asarray(image['rt_values'], dtype=float)
    mz_edges = np.asarray(image['mz_edges'], dtype=float)
    mz_bin = float(image.get('mz_bin', 0.01))
    n_scans, n_mz = grid.shape if grid.ndim == 2 else (0, 0)
    mz_centers = (mz_edges[:-1] + mz_edges[1:]) / 2 if len(mz_edges) >= 2 else np.zeros(0)

    if title is None:
        title = f'DeepIso LC-MS image ({n_scans} scans x {n_mz} m/z bins @ {mz_bin} m/z)'

    # Resolve the optional [n_scans x n_mz_bins] framing window into a rectangle.
    frame_rect = None
    if frame and n_scans and n_mz:
        opts = frame if isinstance(frame, dict) else {}
        fr_scans = int(opts.get('n_scans', 15))
        fr_bins = int(opts.get('n_mz_bins', 211))
        if grid.size:
            peak_row, peak_col = np.unravel_index(int(np.argmax(grid)), grid.shape)
        else:
            peak_row, peak_col = 0, 0
        row0 = int(opts.get('row0', peak_row - fr_scans // 2))
        col0 = int(opts.get('col0', peak_col - fr_bins // 2))
        row0 = int(np.clip(row0, 0, max(0, n_scans - 1)))
        col0 = int(np.clip(col0, 0, max(0, n_mz - 1)))
        row1 = min(n_scans - 1, row0 + fr_scans - 1)
        col1 = min(n_mz - 1, col0 + fr_bins - 1)
        frame_rect = {
            'y0': rt_values[row0], 'y1': rt_values[row1],
            'x0': mz_edges[col0], 'x1': mz_edges[col1 + 1],
            'n_scans': fr_scans, 'n_mz_bins': fr_bins,
        }

    try:
        import plotly.graph_objects as go
        fig = go.Figure(data=go.Heatmap(
            z=grid,
            x=mz_centers,
            y=rt_values,
            colorscale='Greys',
            reversescale=True,
            zmin=0, zmax=255,
            colorbar=dict(title='pixel (0-255)'),
            hovertemplate='m/z: %{x:.3f}<br>RT: %{y:.3f} min<br>pixel: %{z}<extra></extra>',
        ))
        if frame_rect is not None:
            fig.add_shape(
                type='rect',
                x0=frame_rect['x0'], x1=frame_rect['x1'],
                y0=frame_rect['y0'], y1=frame_rect['y1'],
                line=dict(color='#D32F2F', width=2),
                fillcolor='rgba(0,0,0,0)',
            )
            fig.add_annotation(
                x=frame_rect['x1'], y=frame_rect['y1'],
                text=f"[{frame_rect['n_scans']} x {frame_rect['n_mz_bins']}] frame",
                showarrow=False, font=dict(color='#D32F2F', size=11),
                xanchor='right', yanchor='bottom',
            )
        fig.update_layout(
            title=title,
            xaxis_title='m/z',
            yaxis_title='Retention time (min)',
            plot_bgcolor='white',
        )
        return fig
    except Exception:
        import logging
        logging.getLogger(__name__).warning(
            'Plotly unavailable for plot_lcms_heatmap; falling back to matplotlib'
        )
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8, 6))
        if grid.size and len(mz_edges) >= 2 and len(rt_values):
            extent = [mz_edges[0], mz_edges[-1], rt_values[-1], rt_values[0]]
            ax.imshow(grid, aspect='auto', cmap='gray_r', vmin=0, vmax=255,
                      extent=extent, interpolation='nearest')
        if frame_rect is not None:
            import matplotlib.patches as mpatches
            ax.add_patch(mpatches.Rectangle(
                (frame_rect['x0'], min(frame_rect['y0'], frame_rect['y1'])),
                frame_rect['x1'] - frame_rect['x0'],
                abs(frame_rect['y1'] - frame_rect['y0']),
                fill=False, edgecolor='#D32F2F', linewidth=2,
            ))
        ax.set_xlabel('m/z')
        ax.set_ylabel('Retention time (min)')
        ax.set_title(title)
        fig.tight_layout()
        return fig

In [ ]:
def plot_lcms_point_cloud(cloud, title=None):
    """Plot a PointIso LC-MS point cloud as a 3D (m/z, RT, intensity) scatter.

    Renders the arbitrary-precision triplets produced by
    :func:`lcms_point_cloud`: one marker per real peak at its measured
    ``(m/z, retention time, intensity)``, with no binning. Markers are colored by
    intensity. This is the visual counterpart to :func:`plot_lcms_heatmap` -- the
    same feature, stored as sparse points instead of a dense grid. Prefers Plotly
    (interactive 3D), falling back to a matplotlib 3D scatter.

    Parameters
    ----------
    cloud : numpy.ndarray
        ``(n_points, 3)`` array of ``[rt, mz, intensity]`` rows, as returned by
        :func:`lcms_point_cloud`.
    title : str, optional
        Plot title. Defaults to a description including the point count.

    Returns
    -------
    plotly.graph_objs.Figure or matplotlib.figure.Figure
        Interactive Plotly 3D scatter if Plotly is available, else matplotlib.
    """
    cloud = np.asarray(cloud, dtype=float)
    if cloud.ndim != 2 or cloud.shape[1] != 3:
        cloud = cloud.reshape(-1, 3) if cloud.size else np.zeros((0, 3))
    rt, mz, intensity = cloud[:, 0], cloud[:, 1], cloud[:, 2]

    if title is None:
        title = f'PointIso point cloud ({len(cloud)} real peaks)'

    try:
        import plotly.graph_objects as go
        fig = go.Figure(data=go.Scatter3d(
            x=mz, y=rt, z=intensity,
            mode='markers',
            marker=dict(
                size=3,
                color=intensity,
                colorscale='Viridis',
                colorbar=dict(title='intensity'),
                opacity=0.85,
            ),
            hovertemplate='m/z: %{x:.4f}<br>RT: %{y:.3f} min<br>intensity: %{z:.0f}<extra></extra>',
        ))
        fig.update_layout(
            title=title,
            scene=dict(
                xaxis_title='m/z',
                yaxis_title='Retention time (min)',
                zaxis_title='intensity',
            ),
        )
        return fig
    except Exception:
        import logging
        logging.getLogger(__name__).warning(
            'Plotly unavailable for plot_lcms_point_cloud; falling back to matplotlib'
        )
        import matplotlib.pyplot as plt
        from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        if len(cloud):
            sc = ax.scatter(mz, rt, intensity, c=intensity, cmap='viridis', s=8)
            fig.colorbar(sc, ax=ax, shrink=0.6, label='intensity')
        ax.set_xlabel('m/z')
        ax.set_ylabel('Retention time (min)')
        ax.set_zlabel('intensity')
        ax.set_title(title)
        fig.tight_layout()
        return fig